In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, classification_report, roc_curve
)
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
from imblearn.over_sampling import SMOTE
import joblib
import os

RANDOM_STATE = 42
os.makedirs('outputs', exist_ok=True)

print('All imports successful.')

All imports successful.


In [3]:
# Load cleaned CKD dataset
ckd_df = pd.read_csv('../data/chronic_kidney_disease/ckd_cleaned.csv')

print('Dataset loaded successfully')
print('Shape:', ckd_df.shape)
print('\nColumn names:')
print(ckd_df.columns.tolist())
ckd_df.head()

Dataset loaded successfully
Shape: (400, 25)

Column names:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification']


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,48.0,80.0,1.020,1.0,0.0,0.0,0.0,0.0,0.0,121.0,...,44.0,7800.0,5.2,1.0,1.0,0.0,0.0,0.0,0.0,1
1,7.0,50.0,1.020,4.0,0.0,0.0,0.0,0.0,0.0,121.0,...,38.0,6000.0,4.8,0.0,0.0,0.0,0.0,0.0,0.0,1
2,62.0,80.0,1.010,2.0,3.0,0.0,0.0,0.0,0.0,423.0,...,31.0,7500.0,4.8,0.0,1.0,0.0,1.0,0.0,1.0,1
3,48.0,70.0,1.005,4.0,0.0,0.0,1.0,1.0,0.0,117.0,...,32.0,6700.0,3.9,1.0,0.0,0.0,1.0,1.0,1.0,1
4,51.0,80.0,1.010,2.0,0.0,0.0,0.0,0.0,0.0,106.0,...,35.0,7300.0,4.6,0.0,0.0,0.0,0.0,0.0,0.0,1


In [4]:
# Quick sanity check — missing values and data types
print('Missing values:')
print(ckd_df.isnull().sum())
print('\nData types:')
print(ckd_df.dtypes)
print('\nTarget distribution (classification):')
print(ckd_df['classification'].value_counts())

Missing values:
age               0
bp                0
sg                0
al                0
su                0
rbc               0
pc                0
pcc               0
ba                0
bgr               0
bu                0
sc                0
sod               0
pot               0
hemo              0
pcv               0
wc                0
rc                0
htn               0
dm                0
cad               0
appet             0
pe                0
ane               0
classification    0
dtype: int64

Data types:
age               float64
bp                float64
sg                float64
al                float64
su                float64
rbc               float64
pc                float64
pcc               float64
ba                float64
bgr               float64
bu                float64
sc                float64
sod               float64
pot               float64
hemo              float64
pcv               float64
wc                float64
rc              

In [5]:
# Separate features (X) and target (y)
X = ckd_df.drop(columns=['classification'])
y = ckd_df['classification']  # 0 = not CKD, 1 = CKD

print('Features shape:', X.shape)
print('Target shape:', y.shape)
print('\nClass distribution:')
print(y.value_counts())
print(f'\nClass imbalance ratio — Not CKD : CKD = {y.value_counts()[0]}:{y.value_counts()[1]}')

Features shape: (400, 24)
Target shape: (400,)

Class distribution:
classification
1    250
0    150
Name: count, dtype: int64

Class imbalance ratio — Not CKD : CKD = 150:250


In [6]:
# Train-test split (stratified to maintain class distribution)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y          # keeps 250:150 ratio in both splits
)

print(f'Train : {X_train.shape[0]} samples')
print(f'Test  : {X_test.shape[0]} samples')
print('\nTrain class distribution:')
print(y_train.value_counts())
print('\nTest class distribution:')
print(y_test.value_counts())

Train : 320 samples
Test  : 80 samples

Train class distribution:
classification
1    200
0    120
Name: count, dtype: int64

Test class distribution:
classification
1    50
0    30
Name: count, dtype: int64
